In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)


Loading Cleaned Data

In [18]:
df = pd.read_csv("../data/processed/cleaned_data.csv")

df = df.sort_values(
    by=["match_date", "match_id", "innings", "over_number", "ball_number"]
)

print("Dataset shape:", df.shape)
df.head()


C:\Users\G ANBALAGAN\AppData\Local\Temp\ipykernel_20028\1380532071.py:1: DtypeWarning: Columns (40) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/cleaned_data.csv")


Dataset shape: (278205, 52)


,season_id_x,match_id,batter,bowler,non_striker,team_batting,team_bowling,over_number,ball_number,batter_runs,extras,total_runs,batsman_type,bowler_type,player_out,fielders_involved,is_wicket,is_wide_ball,is_no_ball,is_leg_bye,is_bye,is_penalty,wide_ball_runs,no_ball_runs,leg_bye_runs,bye_runs,penalty_runs,wicket_kind,is_super_over,innings,season_id_y,balls_per_over,city,match_date,event_name,match_number,gender,match_type,format,overs,season,team_type,venue,toss_winner,team1,team2,toss_decision,match_winner,win_by_runs,win_by_wickets,player_of_match,result
0,2008,335982,SC Ganguly,P Kumar,BB McCullum,6,1,0,0,0,1,1,Left hand Bat,Right arm Medium,NaN,NaN,False,False,False,True,False,False,0,0,1,0,0,NaN,False,1,2008,6,Bangalore,2008-04-18,Indian Premier League,1.0,male,T20,T20,20,2008,club,M Chinnaswamy Stadium,1,1,6,field,6,140.0,NaN,46.0,win
1,2008,335982,BB McCullum,P Kumar,SC Ganguly,6,1,0,1,0,0,0,Right hand Bat,Right arm Medium,NaN,NaN,False,False,False,False,False,False,0,0,0,0,0,NaN,False,1,2008,6,Bangalore,2008-04-18,Indian Premier League,1.0,male,T20,T20,20,2008,club,M Chinnaswamy Stadium,1,1,6,field,6,140.0,NaN,46.0,win
2,2008,335982,BB McCullum,P Kumar,SC Ganguly,6,1,0,2,0,1,1,Right hand Bat,Right arm Medium,NaN,NaN,False,True,False,False,False,False,1,0,0,0,0,NaN,False,1,2008,6,Bangalore,2008-04-18,Indian Premier League,1.0,male,T20,T20,20,2008,club,M Chinnaswamy Stadium,1,1,6,field,6,140.0,NaN,46.0,win
3,2008,335982,BB McCullum,P Kumar,SC Ganguly,6,1,0,3,0,0,0,Right hand Bat,Right arm Medium,NaN,NaN,False,False,False,False,False,False,0,0,0,0,0,NaN,False,1,2008,6,Bangalore,2008-04-18,Indian Premier League,1.0,male,T20,T20,20,2008,club,M Chinnaswamy Stadium,1,1,6,field,6,140.0,NaN,46.0,win
4,2008,335982,BB McCullum,P Kumar,SC Ganguly,6,1,0,4,0,0,0,Right hand Bat,Right arm Medium,NaN,NaN,False,False,False,False,False,False,0,0,0,0,0,NaN,False,1,2008,6,Bangalore,2008-04-18,Indian Premier League,1.0,male,T20,T20,20,2008,club,M Chinnaswamy Stadium,1,1,6,field,6,140.0,NaN,46.0,win


Aggregate Ball-by-Ball → Batter-Match Level

In [6]:
batter_match = (
    df.groupby(
        ["season", "match_id", "match_date", "batter", "team_batting", "team_bowling", "venue"]
    )
    .agg(
        runs=("batter_runs", "sum"),
        balls_faced=("ball_number", "count"),
        fours=("batter_runs", lambda x: (x == 4).sum()),
        sixes=("batter_runs", lambda x: (x == 6).sum()),
        dismissals=("is_wicket", "sum")
    )
    .reset_index()
)

batter_match.head()


,season,match_id,match_date,batter,team_batting,team_bowling,venue,runs,balls_faced,fours,sixes,dismissals
0,2008,335982,2008-04-18,AA Noffke,1,6,M Chinnaswamy Stadium,9,12,1,0,1
1,2008,335982,2008-04-18,B Akhil,1,6,M Chinnaswamy Stadium,0,2,0,0,1
2,2008,335982,2008-04-18,BB McCullum,6,1,M Chinnaswamy Stadium,158,77,10,13,0
3,2008,335982,2008-04-18,CL White,1,6,M Chinnaswamy Stadium,6,10,0,0,1
4,2008,335982,2008-04-18,DJ Hussey,6,1,M Chinnaswamy Stadium,12,12,1,0,1


Strike Rate & Boundary Ratio

In [7]:
batter_match["strike_rate"] = (
    (batter_match["runs"] / batter_match["balls_faced"]) * 100
).replace([np.inf, -np.inf], 0)

batter_match["boundary_ratio"] = (
    (batter_match["fours"] + batter_match["sixes"]) /
    (batter_match["balls_faced"] + 1)
)


Career Statistics

In [8]:
batter_match["career_runs"] = (
    batter_match.groupby("batter")["runs"].cumsum()
)

batter_match["career_matches"] = (
    batter_match.groupby("batter").cumcount() + 1
)

batter_match["career_avg"] = (
    batter_match["career_runs"] / batter_match["career_matches"]
)


Recent Form(Rolling Averages)

In [9]:
for window in [3, 5, 10]:
    batter_match[f"recent_{window}_avg"] = (
        batter_match
        .groupby("batter")["runs"]
        .rolling(window, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
    )


Consistency & Momentum

In [23]:
batter_match["runs_std_5"] = (
    batter_match
    .groupby("batter")["runs"]
    .rolling(5, min_periods=1)
    .std()
    .reset_index(level=0, drop=True)
)

batter_match["consistency_score"] = (
    batter_match["recent_5_avg"] / (batter_match["runs_std_5"] + 1)
)
batter_match["consistency_score"] = batter_match["consistency_score"].fillna(0)



In [24]:
batter_match["momentum"] = (
    batter_match["recent_3_avg"] - batter_match["recent_10_avg"]
)


Venue & Opponent (PvT) Features

In [25]:
venue_avg = (
    batter_match.groupby("venue")["runs"]
    .transform("mean")
)

batter_match["venue_adjusted_runs"] = (
    batter_match["runs"] / (venue_avg + 1)
)


In [26]:
# Player vs Team (PvT)
batter_match["pvt_avg"] = (
    batter_match
    .groupby(["batter", "team_bowling"])["runs"]
    .transform("mean")
)


Experience Feature

In [27]:
batter_match["experience_log"] = np.log1p(
    batter_match["career_matches"]
)


Target Variable(Next-Match Runs)

In [28]:
batter_match["target_next_runs"] = (
    batter_match
    .groupby("batter")["runs"]
    .shift(-1)
)

batter_match = batter_match.dropna(subset=["target_next_runs"])


Final Feature Dataset

In [29]:
features = [
    "career_avg",
    "recent_3_avg",
    "recent_5_avg",
    "recent_10_avg",
    "strike_rate",
    "boundary_ratio",
    "consistency_score",
    "momentum",
    "venue_adjusted_runs",
    "pvt_avg",
    "experience_log"
]

X = batter_match[features]
y = batter_match["target_next_runs"]

X.head(), y.head()


(   career_avg  recent_3_avg  recent_5_avg  recent_10_avg  strike_rate  \
 1         0.0           0.0           0.0            0.0     0.000000   
 2       158.0         158.0         158.0          158.0   205.194805   
 3         6.0           6.0           6.0            6.0    60.000000   
 4        12.0          12.0          12.0           12.0   100.000000   
 5         8.0           8.0           8.0            8.0   114.285714   
 
    boundary_ratio  consistency_score  momentum  venue_adjusted_runs  \
 1        0.000000                0.0       0.0             0.000000   
 2        0.294872                0.0       0.0             7.344011   
 3        0.000000                0.0       0.0             0.278886   
 4        0.076923                0.0       0.0             0.557773   
 5        0.125000                0.0       0.0             0.371849   
 
      pvt_avg  experience_log  
 1   0.000000        0.693147  
 2  37.933333        0.693147  
 3  15.833333        0.6

In [30]:
final_df = batter_match[features + ["target_next_runs"]]
final_df.to_csv("../data/processed/features.csv", index=False)

print("✅ Feature engineering completed and saved.")


✅ Feature engineering completed and saved.
